In [1]:
```python
# Install if needed:
# %pip install pandas numpy matplotlib scikit-learn vaderSentiment

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import TfidfVectorizer
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

# 1. Load dataset
df = pd.read_csv("Atlantic_Sapphire_vs_Sanford_Stock_Comparison.csv")
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")
print(df.head())

# 2. K-Means clustering on numeric data
numeric = df.select_dtypes(include=np.number)
numeric = numeric.loc[:, numeric.nunique() > 1].replace([np.inf, -np.inf], np.nan)

if numeric.shape[1] >= 2 and len(df) >= 3:
    X = SimpleImputer(strategy="median").fit_transform(numeric)
    X = StandardScaler().fit_transform(X)
    k = min(3, len(df) - 1)
    df["cluster"] = KMeans(
        n_clusters=k, random_state=42, n_init=10
    ).fit_predict(X)

    plt.scatter(X[:, 0], X[:, 1], c=df["cluster"], cmap="viridis")
    plt.title("Stock Data: K-Means Clusters")
    plt.xlabel(numeric.columns[0])
    plt.ylabel(numeric.columns[1])
    plt.show()
else:
    print("Not enough numeric features for K-Means.")

# 3. NLP: find text columns
text_cols = df.select_dtypes(include=["object", "string"]).columns.tolist()

if text_cols:
    df["text"] = df[text_cols].fillna("").astype(str).agg(" ".join, axis=1)
    texts = df["text"].str.strip()
    valid = texts.str.len() > 0

    if valid.sum() >= 2:
        tfidf = TfidfVectorizer(stop_words="english", max_features=1000)
        matrix = tfidf.fit_transform(texts[valid])

        print("Top NLP terms:",
              tfidf.get_feature_names_out()[
                  matrix.sum(axis=0).A1.argsort()[::-1][:10]
              ])

        analyzer = SentimentIntensityAnalyzer()
        df.loc[valid, "sentiment"] = texts[valid].apply(
            lambda t: analyzer.polarity_scores(t)["compound"]
        )
        print(df[["text", "sentiment"]].head())
    else:
        print("Not enough text for NLP.")
else:
    print("No text columns found. NLP requires news or commentary text.")

# 4. Save results
df.to_csv("Atlantic_Sapphire_Sanford_Results.csv", index=False)
print("Analysis complete. Results saved.")
```

SyntaxError: invalid syntax (65495196.py, line 1)